# **Projeto Inteligência Artificial - 2026/2027**
Realizado por: Gonçalo Libânio 55925 & Rodrigo Rosa 55906

## **FASE 1 - PEAS**

## **FASE 2 - Leitura, Auditoria, Limpeza e Visualização**

### 2.1 - Leitura

In [3]:
import os 
import pandas as pd
import numpy as np

folder = os.path.join(os.getcwd(), "data")
SEED = 42

def read_data(file):
    return pd.read_csv(os.path.join(folder, file))

treino = read_data("treino.csv")
validacao = read_data("validacao.csv")
teste = read_data("teste.csv")
rotulos_teste = read_data("rotulos_teste.csv") #ler para juntar com data set do teste pois faltavam os rótulos

teste_merge = teste.merge(
    rotulos_teste,
    on='id_pedido',
    how='left',
    validate='one_to_one'
)

treino.dtypes

id_pedido                     str
equipamento                   str
contexto                      str
utilizadores_afetados     float64
interrupcao_total             str
alternativa_disponivel        str
horas_desde_deteccao      float64
reincidencias_30d         float64
idade_anos                float64
sinal_seguranca               str
periodo                       str
urgente                     int64
dtype: object

### 2.2 - Auditoria & Limpeza

### 2.3 - Visualização

## **FASE 3 - Pré-Processamento**

### 3.1 Pipeline

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train = treino.drop(columns=['id_pedido', 'urgente']) #Características (features)
y_train = treino['urgente'] #Variável alvo (target)

X_val = validacao.drop(columns=['id_pedido', 'urgente'])
y_val = validacao['urgente']

# Identificação dos tipos de atributos
numericas = X_train.select_dtypes(include=[np.number]).columns.tolist()
categoricas = X_train.select_dtypes(include=['str']).columns.tolist()

def make_pipeline(modelo, escalar=False):

    # Preparação numérica
    etapas_num = [
        ('imputer', SimpleImputer(strategy='median'))
    ]

    if escalar:
        etapas_num.append(('scaler', StandardScaler()))

    pipeline_num = Pipeline(etapas_num)

    # Preparação categórica
    pipeline_cat = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Combinar as transformações
    preprocessor = ColumnTransformer([
        ('num', pipeline_num, numericas),
        ('cat', pipeline_cat, categoricas)
    ])

    # Pipeline final
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])

### 3.2 - Métricas de avaliação

In [5]:
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)

def evaluate_model(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    tn, fp, fn, tp = cm.ravel()

    n = len(y_true)

    return {
        'confusion_matrix': cm,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred),
        'taxa_sinalização': (tp + fp) / n, #proporção de pedidos sinalizados como urgentes
        'custo_medio': (4 * fn + fp) / n #custo médio dos erros do modelo
    }